# 04 · Orquestación — un Job con tareas dependientes

**Pregunta que se contesta aquí:** ¿cómo se ejecuta todo el pipeline, en el orden correcto y sin intervención manual?

## Qué pasa en este notebook

Hasta ahora cada notebook se corrió a mano. Aquí se arma un **Job** de Databricks que los ejecuta en cadena, usando el SDK de Python:

```
transformar_silver  →  aplicar_devoluciones  →  construir_gold
 (Bronze → Silver)       (MERGE + OPTIMIZE)      (métricas y features)
```

El orden no sale de la posición de las tareas en la lista, sino de `depends_on`: cada tarea declara de quién depende y solo corre cuando esa tarea terminó con éxito. Por defecto (`run_if = ALL_SUCCESS`), si una tarea falla, las que dependen de ella quedan en estado *upstream failed* y no se ejecutan.


**En esta celda** se hereda la configuración compartida.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importa el SDK de Databricks y se crea el cliente que habla con el workspace. Si el `import` falla, hay que instalar el paquete con `%pip install databricks-sdk -q` en una celda aparte.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
print(f"Workspace: {w.config.host}")

**En esta celda** se calcula la ruta del repositorio en el workspace a partir de la ruta de este mismo notebook, para que el código funcione sin importar dónde esté montado el Git folder. Este notebook vive en la raíz del repositorio, así que se quita solo el último segmento de la ruta.

In [0]:
_current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
BASE_PATH = "/".join(_current_path.split("/")[:-1])
print(f"Repositorio montado en: {BASE_PATH}")

**En esta celda** se crea el Job, solo si no existe ya. Tiene tres tareas encadenadas con `depends_on`, sin cluster (Serverless) y con el schedule pausado.

In [0]:
JOB_NAME = f"[{ENTORNO}] fashion_returns_bronze_a_gold"

existentes = list(w.jobs.list(name=JOB_NAME))
if existentes:
    job_id = existentes[0].job_id
    print(f"[INFO] El Job ya existe — job_id = {job_id}. No se crea uno nuevo.")
else:
    nuevo_job = w.jobs.create(
        name=JOB_NAME,
        tasks=[
            jobs.Task(
                task_key="transformar_silver",
                notebook_task=jobs.NotebookTask(notebook_path=f"{BASE_PATH}/01_Exploracion_y_Transformacion.ipynb"),
            ),
            jobs.Task(
                task_key="aplicar_devoluciones",
                depends_on=[jobs.TaskDependency(task_key="transformar_silver")],
                notebook_task=jobs.NotebookTask(notebook_path=f"{BASE_PATH}/03_Delta_Lake.ipynb"),
            ),
            jobs.Task(
                task_key="construir_gold",
                depends_on=[jobs.TaskDependency(task_key="aplicar_devoluciones")],
                notebook_task=jobs.NotebookTask(notebook_path=f"{BASE_PATH}/03b_Gold_Metricas_y_Features.ipynb"),
            ),
        ],
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 5 * * ?",   # Quartz: seg min hora dia mes dia-semana -> 5:00:00 a. m. todos los dias
            timezone_id="America/Bogota",
            pause_status=jobs.PauseStatus.PAUSED,    # decidido y documentado, pero sin activar
        ),
        tags={"proyecto": "fashion-returns-lakehouse"},
    )
    job_id = nuevo_job.job_id
    print(f"[OK] Job creado — job_id = {job_id}")

print("3 tasks encadenadas: transformar_silver -> aplicar_devoluciones -> construir_gold")
print("Schedule: diario 5:00 AM America/Bogota (PAUSADO)")
print(f"URL: {w.config.host}/jobs/{job_id}")

**En esta celda** se ejecuta el Job una vez y se espera a que termine, para dejar la evidencia de que corrió con las tareas en verde. Si tu versión del SDK no admite `.result()`, ejecuta el Job desde la interfaz (botón *Run now* en la URL de arriba) y revisa que las tres tareas terminen en verde.

In [0]:
ejecucion = w.jobs.run_now(job_id=job_id).result()     # espera a que termine

print(f"Estado final del Job: {ejecucion.state.result_state}")
for t in ejecucion.tasks:
    print(f"  {t.task_key:<24} {t.state.result_state}")

Orquestación · Job de tareas dependientes